In [ ]:
import torch

x = torch.tensor(3.3, requires_grad=True)
y__x = x**2
y__x.retain_grad() #how to get grad of intermediate tensors
f__y = y__x**3
f__y.backward()
display(y__x.grad)
display(x.grad)



# we do backward on f(inputs) where inputs can have other inputs like inputs = y(x)
# backward traverses the computation graph in reverse and applies the chain rule at every node
# so for f(y(x)): df/dy is computed at y(x), and then df/dx = (df/dy)*(dy/dx) is accumulated at x
# to get df/dy we call .grad at y(x) level, and to get df/dx (chain rule already applied) we call .grad at x level

In [36]:
# inplace operation doesn't work well with gradients

In [ ]:
import torch
import random
# training loop of neural network

# setup (initlize input hidden and output layers)
# compute with intialized weights
# compute loss function
# compute gradient and based on that adjust weight
# repeat from step 3 wight updated weights

w = torch.tensor(100.0, requires_grad=True)
b = torch.tensor(100.0, requires_grad=True)
random.seed(34)
n = 5
inp = torch.tensor([[i] for i in range(n)], dtype=torch.float)
out = torch.tensor([[(7)*i + (43)] for i in range(n)])




epochs = 1000
learning_rate = 0.01
for epoch in range(epochs):
    pred =  w*inp + b
    # print(pred)
    # print(out)
    loss = torch.mean(torch.abs(pred - out))   
    loss.backward()
    # print(loss)
    with torch.no_grad():
        # print(w.grad*learning_rate)
        w -= w.grad*learning_rate
        # print(b.grad*learning_rate)
        b -= b.grad*learning_rate
     # Clear gradients
    w.grad.zero_()
    b.grad.zero_()
    # print()
        
    print("W: ", w, "B: ", b)


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(34)
n = 5
inp = torch.tensor([[i] for i in range(n)], dtype=torch.float)
out = torch.tensor([[(7)*i + (43)] for i in range(n)], dtype=torch.float)

model = nn.Linear(1, 1, dtype=torch.float)

display(model.weight)
display(model.bias)
display(out.dtype)

pred = model(inp)
display(pred.dtype)

def train(model, criterion, optimizer, epoches=10000):
    for epoch in range(epoches):
        pred = model(inp)
        # print("keshav")
        # print(out, pred)
        loss = criterion(out, pred)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        

train(model, criterion=nn.MSELoss(), optimizer=torch.optim.SGD(model.parameters(), 0.01))
pred = model(inp)
print(model.weight)
print(out, pred)

# optmizer and model are indirectly linked, optmizer can change model weights(i.e. learning) using the iterator model.parameter provides


In [39]:
# Load training and test datasets
from torchvision import datasets, transforms
# True to download again
train_dataset = datasets.MNIST(root='./data', train=True, download=False)
test_dataset = datasets.MNIST(root='./data', train=False, download=False)


In [ ]:
from torchvision import datasets, transforms
# True to download again
train_dataset = datasets.MNIST(root='./data', train=True, download=False)
test_dataset = datasets.MNIST(root='./data', train=False, download=False)

def preprocess_data():
    train_dataset.data = train_dataset.data.reshape(-1, 784)
    test_dataset.data = test_dataset.data.reshape(-1, 784)
preprocess_data()

print(train_dataset.data[0])
print(test_dataset.data[0])


In [41]:
# print(train_dataset.targets[0])
# train_dataset.targets = torch.tensor([[0 if i!=j else 1 for i in range(10)] for j in train_dataset.targets])
# test_dataset.targets = torch.tensor([[0 if i!=j else 1 for i in range(10)] for j in test_dataset.targets])

# print(train_dataset.targets[0])

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets

train_dataset = datasets.MNIST(root='./data', train=True, download=False)
test_dataset = datasets.MNIST(root='./data', train=False, download=False)
train_dataset.data = train_dataset.data.reshape(-1, 784)
test_dataset.data = test_dataset.data.reshape(-1, 784)

model = nn.Sequential(
    nn.Linear(784, 50),  # input layer:  784 → 50
    nn.ReLU(),           # ✓ no arguments
    nn.Linear(50, 10),   # output layer: 50  → 10 (one per digit)
)

criterion = nn.CrossEntropyLoss()
# Softmax → converts raw scores (logits) into probabilities
# Negative Log Likelihood → penalizes wrong predictions

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
# arg1 = torch.randn(1, 10)
# print(arg1)
# print(criterion(arg1, torch.tensor([5])))


print(train_dataset.data[0])
train_dataset.data = train_dataset.data.float() / 255.0
test_dataset.data  = test_dataset.data.float() / 255.0
print(train_dataset.data[0])
train_dataset.targets = train_dataset.targets.long()
test_dataset.targets  = test_dataset.targets.long()


def train(model, criterion, optimizer, epoches=10000):
    min_validation_error = float("inf")

    patience = 20
    counter = 0

    for epoch in range(epoches):
        optimizer.zero_grad()
        pred = model(train_dataset.data)
        # print(out, pred)
        loss = criterion(pred, train_dataset.targets)
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            test_pred = model(test_dataset.data)
            validation_error = criterion(test_pred, test_dataset.targets)
        if validation_error < min_validation_error:
            min_validation_error = validation_error
            counter = 0
        else:
            counter += 1

        if counter >= patience:
            break

        print(f"epoch {epoch}/{epoches}: loss is {loss.item():.6f} | validation_error is {validation_error.item()}")

    print(f"epoch {epoch}/{epoches}: loss is {loss.item():.6f} | validation_error is {validation_error.item()}")


train(model, criterion, optimizer)


In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets

train_dataset = datasets.MNIST(root='./data', train=True, download=False)
test_dataset = datasets.MNIST(root='./data', train=False, download=False)
train_dataset.data = train_dataset.data.reshape(-1, 784).float() / 255.0
test_dataset.data = test_dataset.data.reshape(-1, 784).float() / 255.0
train_dataset.targets = train_dataset.targets.long()
test_dataset.targets = test_dataset.targets.long()

model = nn.Sequential(
    nn.Linear(784, 50),
    nn.ReLU(),
    nn.Linear(50, 10),
)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def train(model, criterion, optimizer, epoches=10000):
    min_validation_error = float("inf")
    patience = 20
    counter = 0
    for epoch in range(epoches):
        optimizer.zero_grad()
        pred = model(train_dataset.data)
        loss = criterion(pred, train_dataset.targets)
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            test_pred = model(test_dataset.data)
            validation_error = criterion(test_pred, test_dataset.targets)
        if validation_error < min_validation_error:
            min_validation_error = validation_error
            counter = 0
        else:
            counter += 1
        if counter >= patience:
            break

train(model, criterion, optimizer)

test_pred = model(test_dataset.data)

criterion(test_pred, test_dataset.targets)

In [ ]:
# 3. Normalize inputs
# Neural nets train better when values are small and centered.

In [ ]:
# Lab 1 — Linear regression with raw tensors + autograd
# Load California housing, split into train/valid/test, and turn each split into float tensors.
# Normalize the features using tensor operations only (no StandardScaler). Compute the statistics on the training set, then apply them everywhere.
# The raw targets come out as 1D. Reshape them so predictions and targets are the same shape. Figure out which shape the math actually wants.
# Create a weight tensor and a bias tensor as trainable parameters. Initialize one randomly and the other at zero.
# Write a full batch gradient descent loop: predict, compute MSE, backprop, update under a no-grad context, reset gradients, print the loss each epoch.
# Use the trained parameters to predict a few new instances.
# Go further: Sweep the learning rate across a few orders of magnitude — find where it diverges, where it crawls, where it's just right. Does initializing the bias randomly instead of at zero change anything here? Why might that change for a deeper network?


from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import torch


try:
    dataset = fetch_california_housing()
except Exception as e:
    raise RuntimeError("failed to load data") from e

print(dataset.feature_names)

data = dataset.data
# print(data)

X_train, X_test, y_train, y_test = train_test_split(data, dataset.target)
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

# print(X_train)
# print(X_test)
# print(y_test)
# print(y_train)

#             what is normalization?
# # x-x_min/(x_max-x_min)
# from sklearn.preprocessing import StandardScaler
# scaler = StandardScaler

# scaler = StandardScaler()

# X_train = scaler.fit_transform(X_train.numpy())
# X_test = scaler.transform(X_test.numpy())

# X_train = torch.tensor(X_train, dtype=torch.float32)
# X_test = torch.tensor(X_test, dtype=torch.float32)
# Compute statistics on the training set only
mean = X_train.mean(dim=0)
std = X_train.std(dim=0)

# Avoid division by zero (good practice)
std[std == 0] = 1.0

# Apply the same transformation to both train and test
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

print(X_train)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train = X_train.to(device)
y_train = y_train.to(device)
y_test = y_test.to(device)
 
X_test = X_test.to(device)



['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
tensor([[-0.8355,  0.1105, -0.5487,  ...,  0.1182, -0.7297,  0.9637],
        [ 0.0935,  0.3491, -0.2052,  ..., -0.1306, -1.3690,  1.1987],
        [-0.5972,  1.8603,  0.4248,  ..., -0.1440, -0.1184,  0.2688],
        ...,
        [ 2.0178, -0.2077,  0.1111,  ...,  0.6613,  0.8336, -1.1611],
        [-1.1888,  0.7468, -0.6954,  ...,  0.1798, -0.7764,  0.6437],
        [-1.0922, -0.6849, -0.3374,  ...,  0.1322, -0.0204,  0.1188]])


In [27]:
# important
# 1. 
class Noun:
    def __init__(self, type="thing"):
        self.type = type

class Bag:
    def __init__(self):
        self.a = 1
        self.b = "hello"
        self.noun = Noun()

bag = Bag()
print(vars(bag))    # {'a': 1, 'b': 'hello'}

# 2.
class Doubler:
    def __call__(self, x):
        return x * 2

f = Doubler()
print(f(10))    # 20 — calling the instance runs __call__

# 3.

class Doubler:
    def __call__(self, x):
        return x * 2

f = Doubler()
print(f(10))    # 20 — calling the instance runs __call__
# super().__init__(...) means: "run my parent class's constructor." If you forget it, the parent's setup never happens. This matters enormously in PyTorch — nn.Module.__init__ sets up the internal bookkeeping that tracks your parameters. Forget super().__init__() in a model and you get a confusing error the moment you try to use it.


# 4.
# Calling .backward() on a scalar walks that record in reverse and computes the derivative of that scalar with respect to every tensor that has requires_grad=True:


# isinstance(nn.CrossEntropyLoss(), nn.Module) → True.

# vars(torch.nn.modules)



{'a': 1, 'b': 'hello', 'noun': <__main__.Noun object at 0x00000290ADCACC80>}
20
20


In [14]:
import torch

# --- fake data: y = 4x + 3, plus noise ---
x = torch.linspace(0, 10, 100).reshape(-1, 1)          # shape (100, 1)
y_true = 4 * x + 3 + torch.randn(100, 1) * 0.5

# --- "model": just two tensors ---
w = torch.randn(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

learning_rate = 0.01

for step in range(1000):
    # 1. forward: compute predictions
    y_pred = x * w + b                                  # broadcasting does the work

    # 2. loss: mean squared error, written out
    loss = ((y_pred - y_true) ** 2).mean()
    print(loss)
    # 3. backward: fills w.grad and b.grad
    loss.backward()
    print(w.grad)
    # 4. update: gradient descent by hand
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
    w.grad.zero_()
    b.grad.zero_()

    if step % 200 == 0:
        print(f"step {step}: loss={loss.item():.4f}, w={w.item():.3f}, b={b.item():.3f}")

# w ends up near 4, b near 3

tensor(402.5222, grad_fn=<MeanBackward0>)
tensor([-231.5355])
step 0: loss=402.5222, w=3.309, b=0.361
tensor(41.5071, grad_fn=<MeanBackward0>)
tensor([-72.7924])
tensor(5.6876, grad_fn=<MeanBackward0>)
tensor([-22.8000])
tensor(2.1215, grad_fn=<MeanBackward0>)
tensor([-7.0565])
tensor(1.7545, grad_fn=<MeanBackward0>)
tensor([-2.0990])
tensor(1.7049, grad_fn=<MeanBackward0>)
tensor([-0.5384])
tensor(1.6869, grad_fn=<MeanBackward0>)
tensor([-0.0475])
tensor(1.6721, grad_fn=<MeanBackward0>)
tensor([0.1065])
tensor(1.6579, grad_fn=<MeanBackward0>)
tensor([0.1544])
tensor(1.6438, grad_fn=<MeanBackward0>)
tensor([0.1689])
tensor(1.6298, grad_fn=<MeanBackward0>)
tensor([0.1728])
tensor(1.6160, grad_fn=<MeanBackward0>)
tensor([0.1735])
tensor(1.6023, grad_fn=<MeanBackward0>)
tensor([0.1731])
tensor(1.5887, grad_fn=<MeanBackward0>)
tensor([0.1724])
tensor(1.5753, grad_fn=<MeanBackward0>)
tensor([0.1716])
tensor(1.5620, grad_fn=<MeanBackward0>)
tensor([0.1708])
tensor(1.5489, grad_fn=<MeanBackwa

In [ ]:
import torch
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

dataset = fetch_california_housing()
X_train, X_test, y_train, y_test = train_test_split(dataset.data, dataset.target)
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

mean = X_train.mean(dim=0)
std = X_train.std(dim=0)
std[std == 0] = 1.0
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

learning_rate = 0.01
class customNeuralNetwork(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = torch.nn.Parameter( #model.parameters() will show this if we use this
            torch.tensor([1.0, 2.0, 3.0, 1.0, 2.0, 1.0, 2.0, 3.0], dtype=torch.float32)
        )
        self.bias = torch.nn.Parameter(torch.tensor(1.0, dtype=torch.float32))

    def forward(self, inputs):
        # inputs shape: [batch_size, 8]
        # output shape: [batch_size]
        return inputs @ self.weights + self.bias


model = customNeuralNetwork()

for step in range(20000):
    model.zero_grad()
    y_train_pred = model(X_train) #this is why we are using torch.nn.modules, to use model(instance of customNeuralNetwork) as a function
    # print(y_train_pred)
    loss = ((y_train_pred - y_train)**2).mean()
    # display(y_train_pred)
    # break
    # print(loss)
    loss.backward()
    with torch.no_grad():
        # in-place update keeps these as nn.Parameter objects; reassigning
        # model.weights = ... would replace the Parameter with a plain Tensor
        model.weights -= learning_rate * model.weights.grad
        model.bias -= learning_rate * model.bias.grad
    
print([parameter for parameter in model.parameters()])


In [ ]:
# Lab 2 — Same model, high-level API
# Replace your hand-rolled weight/bias with a single built-in linear layer. Inspect its weight and bias attributes. Compare the weight's shape to the tensor you made by hand in Lab 1 — they're related by a transpose. Make sure you can explain why.
# Iterate over the module's parameters two ways: anonymously, and as name/value pairs.
# Call the model on a small batch as if it were a function. Look at the result's grad_fn. What does calling the module actually trigger under the hood?
# Attach a hook that fires whenever the module runs, then remove it. Confirm it does not fire if you bypass the normal call path — and form an opinion on why you should never bypass it.
# Swap your manual update + zero-grad lines for an optimizer and a loss-function object. Wrap it all in a reusable training function.
# Train, predict, and compare numbers against Lab 1. They'll be close but not identical — track down the reason.
# Go further: The loss object is also a "module." What else in this library turns out to be a module? Why is that a useful design choice?
import torch
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

dataset = fetch_california_housing()
X_train, X_test, y_train, y_test = train_test_split(dataset.data, dataset.target)
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

mean = X_train.mean(dim=0)
std = X_train.std(dim=0)
std[std == 0] = 1.0
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_train = X_train.to(device)
y_train = y_train.to(device)
X_test = X_test.to(device)
y_test = y_test.to(device)

class customNeuralNetwork(torch.nn.Module):
    def __init__(self):
        super().__init__()
        in_features = 8
        out_features = 1
        self.linear_layer = torch.nn.Linear(in_features, out_features, bias=True, device=None, dtype=None)

    def forward(self, inputs):
        return self.linear_layer(inputs)


model = customNeuralNetwork().to(device)
optimizer = torch.optim.Adam([
    {'params': model.linear_layer.parameters(), 'lr': 1e-3},
])

for step in range(45000):
    model.zero_grad()
    y_train_pred = model(X_train) #this is why we are using torch.nn.modules, to use model(instance of customNeuralNetwork) as a function

    loss = ((y_train_pred - y_train)**2).mean()
    
    loss.backward()
    optimizer.step()
    
    if step % 1000 == 0:
        print(f"step {step}, loss {loss.item():.6f}")
print([parameter for parameter in model.parameters()])

y_test_pred = model(X_test)
loss = ((y_test_pred - y_test)**2).mean()
print(loss)

# Way 1: anonymously — just the tensors
for param in model.parameters():
    print(param.shape, param.requires_grad)

# Way 2: name/value pairs — you can see *what* each tensor is
for name, param in model.named_parameters():
    print(name, param)

# addmmBackward0

In [ ]:
# Lab 3 — Regression MLP
# Stack a few linear layers with activations between them into one sequential model. Make the input width match your features and the output width match your targets, and make every layer's output line up with the next layer's input.
# Train it with the function you built in Lab 2.
# Compare its loss to the plain linear model.
# Go further: Does adding more neurons or more layers keep helping, or does it plateau / get worse? Try making all hidden layers the same width vs. different widths — does it matter?

import torch
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

dataset = fetch_california_housing()
X_train, X_test, y_train, y_test = train_test_split(dataset.data, dataset.target)
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

mean = X_train.mean(dim=0)
std = X_train.std(dim=0)
std[std == 0] = 1.0
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_train = X_train.to(device)
y_train = y_train.to(device)
X_test = X_test.to(device)
y_test = y_test.to(device)

class customNeuralNetwork(torch.nn.Module):
    def __init__(self):
        super().__init__()
        in_features = 8
        out_features = 1
        self.linear_layer = torch.nn.Linear(in_features, 12, bias=True, device=None, dtype=None)
        self.relu_layer = torch.nn.ReLU()
        self.linear_layer_2 = torch.nn.Linear(12, out_features, bias=True, device=None, dtype=None)


    def forward(self, inputs):
        o1 = self.linear_layer(inputs)
        # print(o1)
        o2 = self.relu_layer(o1)
        # print(o2)
        o3 = self.linear_layer_2(o2)
        
        return o3


model = customNeuralNetwork().to(device)
optimizer = torch.optim.Adam([
    {'params': model.linear_layer.parameters(), 'lr': 1e-3},
    {"params": model.linear_layer_2.parameters(), "lr" : 1e-3}

])

for step in range(4500):
    model.zero_grad()
    y_train_pred = model(X_train) #this is why we are using torch.nn.modules, to use model(instance of customNeuralNetwork) as a function

    loss = ((y_train_pred - y_train)**2).mean()
    
    loss.backward()
    optimizer.step()
    
    if step % 1000 == 0:
        print(f"step {step}, loss {loss.item():.6f}")
print([parameter for parameter in model.parameters()])

y_test_pred = model(X_test)
loss = ((y_test_pred - y_test)**2).mean()
print(loss)

# Way 1: anonymously — just the tensors
for param in model.parameters():
    print(param.shape, param.requires_grad)

# Way 2: name/value pairs — you can see *what* each tensor is
for name, param in model.named_parameters():
    print(name, param)

# addmmBackward0

step 0, loss 5.870746
step 1000, loss 2.138024
step 2000, loss 1.557176
step 3000, loss 1.444760
step 4000, loss 1.386337
[Parameter containing:
tensor([[-0.8927,  0.1043, -0.4561, -0.7513, -0.0160,  1.8322, -1.6133,  1.6886],
        [ 0.9318,  0.5794,  1.0031,  0.4315, -0.9942, -1.2006,  0.4965, -0.8128],
        [-1.4660,  2.3464, -1.7683, -0.5318, -0.8876,  0.0242, -0.1989, -0.3930],
        [ 0.3306,  0.5589,  0.5532,  0.3742,  0.3069, -0.3030, -0.3194,  0.0836],
        [ 0.4288, -1.3747,  0.8337, -0.6574, -0.1802, -0.1432, -1.8394,  1.5971],
        [-0.2700, -0.0916, -0.1695, -0.3028, -0.4973, -0.3854, -0.3150,  0.6998],
        [-0.6487, -0.8952, -0.5982, -0.7042,  0.8015, -0.2394,  0.6829, -0.9152],
        [-1.1357, -0.1176, -0.6908, -0.5027, -0.1950,  0.0566, -1.5884,  1.6108],
        [-1.1511, -0.6204, -0.7599, -0.8848,  0.9130, -0.7282, -1.3851,  0.9737],
        [-0.6724, -0.2333,  0.4253, -0.0379,  0.6630,  1.0818, -0.3260,  0.1763],
        [-0.0717,  1.9224,  0.1597,

In [ ]:
# Lab 4 — Mini-batch gradient descent with DataLoaders
# Wrap your training tensors in a dataset object, then feed it through a loader that serves shuffled batches of a fixed size.
# Move the model to your device, and copy each batch to the device inside the loop.
# Write a train() function that: switches the model into training mode, loops epochs, loops batches, and reports the mean loss per epoch.
# Compare convergence and per-epoch wall-clock time against full-batch training.
# Explore the loader speed knobs one at a time and time the effect: pinned memory, non-blocking transfers, multiple worker processes, prefetching, persistent workers.
# Go further: Skip the built-in dataset wrapper and write your own dataset class with just a length method and an item-getter. What's the minimum interface a loader actually needs? When do extra workers slow you down instead of speeding you up?

import torch
from torch.utils.data import DataLoader

class CustomDataset(torch.utils.data.Dataset):
    def __init__(self):
        dataset = fetch_california_housing()
        self.X = dataset.data
        self.Y = dataset.target

    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, index):

        return self.X[index], self.Y[index]
    

dataset = CustomDataset()
dataloader = DataLoader(dataset, batch_size=3, shuffle=True)
print(dataset[0])

for i in range(1):

    for batch_x, batch_y in dataloader:
        print(batch_x, batch_y)
        break

# collate:  this stacking step is called collation
    
    

(array([   8.3252    ,   41.        ,    6.98412698,    1.02380952,
        322.        ,    2.55555556,   37.88      , -122.23      ]), np.float64(4.526))
tensor([[ 3.3500e+00,  3.6000e+01,  4.2854e+00,  9.9495e-01,  1.2740e+03,
          3.2172e+00,  3.3800e+01, -1.1825e+02],
        [ 4.1325e+00,  3.2000e+01,  5.4396e+00,  1.0858e+00,  2.1520e+03,
          2.2797e+00,  3.2640e+01, -1.1707e+02],
        [ 7.6229e+00,  3.5000e+01,  7.2251e+00,  1.0205e+00,  8.8100e+02,
          2.5760e+00,  3.7390e+01, -1.2210e+02]], dtype=torch.float64) tensor([1.6310, 1.7280, 5.0000], dtype=torch.float64)


In [10]:
import torch
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class customNeuralNetwork(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_layer = torch.nn.Linear(8, 12, bias=True)
        self.relu_layer = torch.nn.ReLU()
        self.linear_layer_2 = torch.nn.Linear(12, 1, bias=True)

    def forward(self, inputs):
        o1 = self.linear_layer(inputs)
        o2 = self.relu_layer(o1)
        o3 = self.linear_layer_2(o2)
        return o3


class CustomDataset(Dataset):
    def __init__(self, X, Y):
        self.X = X
        self.Y = Y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.Y[index]


# Load and split BEFORE normalizing, so test stats don't leak into train
data = fetch_california_housing()
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)

X_train = torch.tensor(X_train_raw, dtype=torch.float32)
X_test = torch.tensor(X_test_raw, dtype=torch.float32)
y_train = torch.tensor(y_train_raw, dtype=torch.float32)
y_test = torch.tensor(y_test_raw, dtype=torch.float32)

# Fit normalization stats on TRAIN only, apply to both
mean = X_train.mean(dim=0)
std = X_train.std(dim=0)
std[std == 0] = 1.0
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

X_test = X_test.to(device)
y_test = y_test.to(device)

dataset = CustomDataset(X_train, y_train)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

model = customNeuralNetwork().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for step in range(200):
    total_loss = 0
    count = 0
    for batch_x, batch_y in dataloader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        y_train_pred = model(batch_x).squeeze(1)  # match shapes
        loss = ((y_train_pred - batch_y) ** 2).mean()
        total_loss += loss.detach() #imp: loss.detach() returns a new tensor
        count +=1
        loss.backward()
        optimizer.step()
    mean_loss = total_loss/count

    if step % 20 == 0:
        print(f"step {step}, mean loss {mean_loss.item():.6f}")

model.eval()
with torch.no_grad():
    y_test_pred = model(X_test).squeeze(1)
    test_loss = ((y_test_pred - y_test) ** 2).mean()
print("Test MSE:", test_loss.item())

step 0, mean loss 3.615129
step 20, mean loss 0.378418
step 40, mean loss 0.340806
step 60, mean loss 0.328251
step 80, mean loss 0.323176
step 100, mean loss 0.320428
step 120, mean loss 0.317833
step 140, mean loss 0.318139
step 160, mean loss 0.316477
step 180, mean loss 0.316331
Test MSE: 0.3312635123729706
